# JPMorganChase Quantitative Modeling Tech Screen — Part 3
## Logistic Regression Refresher

This notebook is tailored for a **Model Risk Governance Review / applied data science tech screen**.

You should be able to:
- explain logistic regression simply
- write the complete sklearn workflow from memory
- interpret predicted probabilities
- understand coefficients and odds
- evaluate classification performance
- handle class imbalance
- explain scaling and leakage
- avoid common implementation mistakes
- produce runnable code quickly

### Core workflow
**Choose X/y → split → fit preprocessing on train → transform train/test → fit model → predict probabilities/classes → evaluate → sanity-check**

# 1. What logistic regression does

Logistic regression is a **binary classification model**.

Example target:
- `default = 1` → borrower defaults
- `default = 0` → borrower does not default

It first forms a linear score:

\[
z = \beta_0 + \beta_1x_1 + \cdots + \beta_px_p
\]

Then maps that score to a probability using the sigmoid:

\[
P(Y=1|X) = \frac{1}{1+e^{-z}}
\]

### What to say
> "Logistic regression estimates the probability of a binary outcome. It takes a linear combination of the features and passes it through the logistic function so the output lies between zero and one."

# 2. Probability first, class second

The model produces a probability first.

Example:
- predicted probability of default = `0.73`
- with threshold `0.5`, predict class `1`

The threshold does **not** have to be 0.5.

### What to say
> "The model estimates a probability first. The final class depends on a decision threshold, which can be changed depending on the cost of false positives and false negatives."

# 3. Create a small synthetic credit-default dataset

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

np.random.seed(42)

n = 600

income = np.random.normal(80000, 25000, n).clip(20000, 200000)
loan_amount = np.random.normal(35000, 16000, n).clip(5000, 120000)
credit_score = np.random.normal(690, 55, n).clip(500, 850)
utilization = np.random.beta(2, 4, n)

loan_to_income = loan_amount / income

log_odds = (
    -2.5
    + 3.0 * loan_to_income
    + 2.0 * utilization
    - 0.008 * (credit_score - 650)
)

prob_default = 1 / (1 + np.exp(-log_odds))
default = np.random.binomial(1, prob_default)

df = pd.DataFrame({
    "income": income,
    "loan_amount": loan_amount,
    "credit_score": credit_score,
    "utilization": utilization,
    "loan_to_income": loan_to_income,
    "default": default
})

print(df.head())
print("\nClass balance:")
print(df["default"].value_counts(normalize=True))

ModuleNotFoundError: No module named 'sklearn'

# 4. Separate features and target

`X` = predictors  
`y` = target

### What to say
> "I'll separate the predictors from the target before splitting the dataset."

In [ ]:
X = df.drop(columns=["default"])
y = df["default"]

print(X.head())
print(y.head())

# 5. Train/test split

Use a held-out test set to measure generalization.

For classification, `stratify=y` keeps approximately the same class proportions in train and test.

### What to say
> "I'm holding out a test set so performance is measured on data the model did not fit on. I'll stratify by the target to preserve class balance."

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Train default rate:", y_train.mean())
print("Test default rate :", y_test.mean())

# 6. Scaling

Logistic regression does **not strictly require** standardized features.

Scaling is still useful because:
- predictors may have very different units
- optimization can be more stable
- regularization treats coefficients more comparably

Critical rule:

**Fit the scaler on training data only.**

Correct:
```python
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
```

Incorrect:
```python
X_test = scaler.fit_transform(X_test)
```

### What to say
> "I'll fit the scaler on the training data only and apply the same transformation to the test set. Fitting preprocessing on test data would introduce leakage."

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled[:3])

# 7. Fit logistic regression

This syntax should be automatic.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 8. `predict()` vs `predict_proba()`

`predict()` returns hard classes.

`predict_proba()` returns probabilities.

For binary classification:

```python
model.predict_proba(X_test)[:, 1]
```

gives the estimated probability of class `1`.

### What to say
> "I'll keep the predicted probabilities because metrics such as ROC-AUC use the probability ranking rather than only the final class."

In [ ]:
pred = model.predict(X_test_scaled)
prob = model.predict_proba(X_test_scaled)[:, 1]

print("First 10 predictions:")
print(pred[:10])

print("\nFirst 10 default probabilities:")
print(prob[:10])

# 9. Accuracy

\[
Accuracy = \frac{\text{correct predictions}}{\text{all predictions}}
\]

Useful when classes are reasonably balanced.

Potentially misleading when one class dominates.

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred))

# 10. Confusion matrix

Scikit-learn binary confusion matrix:

```text
[[TN, FP],
 [FN, TP]]
```

For `default = 1`:

- **TN**: correctly predicted non-default
- **FP**: predicted default, borrower actually repays
- **FN**: predicted non-default, borrower actually defaults
- **TP**: correctly predicted default

### What to say
> "The relative importance of false positives and false negatives depends on the business objective. In credit risk, missing an actually risky borrower can be costly, so recall on the default class may matter."

In [ ]:
cm = confusion_matrix(y_test, pred)
print(cm)

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

# 11. Precision and recall

### Precision

\[
Precision = \frac{TP}{TP+FP}
\]

Of everything predicted positive, how many were actually positive?

High precision = fewer false positives.

### Recall

\[
Recall = \frac{TP}{TP+FN}
\]

Of all actual positives, how many did we catch?

High recall = fewer false negatives.

### What to say
> "If defaults are relatively rare, I would not rely on accuracy alone. I would also inspect precision, recall, the confusion matrix, and ROC-AUC."

In [ ]:
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 score :", f1_score(y_test, pred))

# 12. F1 score

\[
F1 = 2\frac{Precision \times Recall}{Precision + Recall}
\]

F1 balances precision and recall.

Do not automatically assume F1 is the correct business objective. The metric should match the cost of model errors.

# 13. ROC-AUC

ROC-AUC measures how well the model ranks positive observations above negative ones across thresholds.

Rule of thumb:
- `0.5` → random ranking
- `1.0` → perfect ranking

Use **probabilities**, not hard predictions.

In [ ]:
auc = roc_auc_score(y_test, prob)
print("ROC-AUC:", auc)

### Common mistake

Less informative:
```python
roc_auc_score(y_test, pred)
```

Correct:
```python
roc_auc_score(y_test, prob)
```

# 14. Classification report

In [ ]:
print(classification_report(y_test, pred))

# 15. Coefficient interpretation

Logistic regression is linear in **log-odds**:

\[
\log\left(\frac{p}{1-p}\right)
=
\beta_0 + \beta_1x_1 + \cdots
\]

Interpretation:
- positive coefficient → increasing feature raises log-odds of class 1
- negative coefficient → increasing feature lowers log-odds of class 1

Because we scaled the predictors, a one-unit change in the model input corresponds to roughly a **one-standard-deviation change** in the original variable.

### What to say
> "A positive coefficient means the feature increases the log-odds of the positive class, holding other variables fixed."

In [ ]:
coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

print(coef_table)

# 16. Odds ratios

Exponentiate a coefficient:

```python
np.exp(beta)
```

If `beta = 0.40`:

```text
exp(0.40) ≈ 1.49
```

The odds are multiplied by about `1.49` for a one-unit increase in the feature, holding other features fixed.

Important:

**This is an odds change, not a direct probability change.**

In [ ]:
coef_table["odds_ratio"] = np.exp(coef_table["coefficient"])
print(coef_table)

# 17. Classification threshold tradeoff

Lower threshold:
- predicts more positives
- recall usually increases
- false positives usually increase
- precision may decrease

### What to say
> "The threshold should reflect the relative cost of classification errors rather than being chosen mechanically."

In [ ]:
threshold = 0.30

pred_30 = (prob >= threshold).astype(int)

print("Threshold:", threshold)
print("Precision:", precision_score(y_test, pred_30))
print("Recall   :", recall_score(y_test, pred_30))
print(confusion_matrix(y_test, pred_30))

# 18. Class imbalance

Suppose:
- 95% non-default
- 5% default

A useless model predicting **non-default for everybody** gets 95% accuracy.

Therefore always inspect:

```python
y.value_counts(normalize=True)
```

Potential responses:
1. use precision / recall / ROC-AUC
2. change threshold
3. use `class_weight="balanced"`
4. consider resampling if appropriate

For this screen, **recognizing the issue** is more important than using complicated techniques.

In [ ]:
print(y.value_counts())
print(y.value_counts(normalize=True))

In [ ]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_model.fit(X_train_scaled, y_train)

balanced_pred = balanced_model.predict(X_test_scaled)
balanced_prob = balanced_model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, balanced_pred))
print("ROC-AUC:", roc_auc_score(y_test, balanced_prob))

# 19. Regularization

Scikit-learn logistic regression uses regularization by default.

Regularization discourages very large coefficients and can reduce overfitting.

`C` is the **inverse** of regularization strength:

- smaller `C` → stronger regularization
- larger `C` → weaker regularization

### What to say
> "Regularization penalizes large coefficients and can help control overfitting. In sklearn, smaller C means stronger regularization."

In [ ]:
strong_reg = LogisticRegression(C=0.1, max_iter=1000)
weak_reg = LogisticRegression(C=10, max_iter=1000)

strong_reg.fit(X_train_scaled, y_train)
weak_reg.fit(X_train_scaled, y_train)

print("Strong-reg coefficient norm:",
      np.linalg.norm(strong_reg.coef_))

print("Weak-reg coefficient norm:",
      np.linalg.norm(weak_reg.coef_))

# 20. L1 vs L2

### L2
Penalizes squared coefficient magnitude.

Usually shrinks coefficients without forcing many exactly to zero.

### L1
Penalizes absolute coefficient magnitude.

Can force coefficients exactly to zero, giving feature-selection behavior.

For this interview, know the concept. Do not over-focus on tuning unless asked.

# 21. Multicollinearity

If you include:
- `loan_amount`
- `income`
- `loan_to_income`

the features are mechanically related.

Prediction may still be fine, but individual coefficient estimates can become unstable.

### What to say
> "If interpretation matters, I would inspect highly correlated or mechanically related features because multicollinearity can make individual coefficients unstable."

In [ ]:
print(df.drop(columns=["default"]).corr())

# 22. Leakage

For a default model, suspicious predictors could include:

- `days_since_default`
- `recovery_amount`
- `post_default_collection_status`

These may only become known after default.

### What to say
> "I would verify that every predictor was observable at the model's decision time. Otherwise performance could be artificially inflated by target leakage."

# 23. Missing values

`LogisticRegression` does not directly accept `NaN`.

Missing values therefore need to be handled before fitting.

In a rigorous workflow, imputation parameters should also be estimated **only from training data**.

That is the same leakage principle as scaling.

# 24. Pipeline — safer implementation

A sklearn `Pipeline` keeps preprocessing and modeling together and helps avoid leakage.

It is good to know, though simple explicit code may be faster to debug in a timed screen.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

pipe_pred = pipe.predict(X_test)
pipe_prob = pipe.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pipe_pred))
print("ROC-AUC :", roc_auc_score(y_test, pipe_prob))

# 25. Full workflow to memorize

You should be able to reproduce this block from a blank editor.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# 1. Features and target
X = df.drop(columns=["default"])
y = df["default"]

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Fit
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 5. Predict
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

# 6. Evaluate
print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC :", roc_auc_score(y_test, prob))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred))

print("\nClassification report:")
print(classification_report(y_test, pred))

# 26. Common interview mistakes

### 1. Scaling before train/test split
This lets test information influence preprocessing.

### 2. Calling `fit_transform()` on test data
Use `transform()` only.

### 3. Using hard predictions for ROC-AUC
Use predicted probabilities.

### 4. Reporting accuracy without checking class balance
A majority-class model can look deceptively strong.

### 5. Treating coefficients as probability changes
Coefficients are changes in **log-odds**.

### 6. Scaling suspicious huge values before investigating
Scaling does not repair bad data.

### 7. Ignoring leakage
A clean feature may still contain future information.

### 8. Overengineering
A simple correct sklearn solution is better than a complex half-working pipeline during a scored coding interview.

# 27. Likely verbal questions

### Why logistic regression?
> "It is simple, interpretable, computationally efficient, and produces probabilities directly, making it a strong baseline for binary outcomes such as default."

### Does logistic regression require scaling?
> "Not strictly, but scaling helps when features have very different magnitudes, especially for optimization and regularization."

### Why not just use accuracy?
> "If the target is imbalanced, a model can achieve high accuracy by predicting the majority class. I would also inspect precision, recall, the confusion matrix, and ROC-AUC."

### `predict` vs `predict_proba`?
> "`predict_proba` returns estimated class probabilities. `predict` converts those into final class labels using the decision rule."

### What happens if you lower the threshold?
> "The model predicts more positives. Recall usually increases, while false positives also tend to increase."

### What does a positive coefficient mean?
> "Holding other variables fixed, increasing that feature increases the log-odds of the positive class."

### What does `exp(beta)` mean?
> "It is the multiplicative change in odds for a one-unit increase in that feature, holding other variables fixed."

### What is regularization doing?
> "It penalizes large coefficients and helps control overfitting. In sklearn, smaller C means stronger regularization."

### What would concern you in model validation?
> "Leakage, unstable coefficients, poor out-of-sample performance, class imbalance, inappropriate preprocessing, implausible relationships, and sensitivity to small changes in assumptions."

# 28. Five-minute recall sheet

## Syntax

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]
```

## Metrics

```python
accuracy_score(y_test, pred)
confusion_matrix(y_test, pred)
classification_report(y_test, pred)
roc_auc_score(y_test, prob)
```

## Concepts to remember

- probability first, class second
- threshold controls precision/recall tradeoff
- split before scaling
- fit preprocessing on train only
- accuracy can mislead under imbalance
- positive beta increases log-odds of class 1
- `exp(beta)` = odds ratio
- smaller `C` = stronger regularization
- check leakage
- simple runnable code > fancy broken code

# 29. Final self-test

You should answer all of these without notes:

1. Why does logistic regression output values between 0 and 1?
2. What does `predict_proba(X)[:, 1]` return?
3. Why `fit_transform` on train but only `transform` on test?
4. What is leakage?
5. What is a false negative if class 1 means default?
6. Precision vs recall?
7. Why can 95% accuracy be useless?
8. What does a positive coefficient mean?
9. What does `exp(beta)` mean?
10. What happens when you lower the classification threshold?
11. What does smaller `C` mean?
12. Why investigate huge values before scaling?
13. Why use `stratify=y`?
14. What should ROC-AUC use: `pred` or `prob`?
15. Can sklearn logistic regression accept `NaN` directly?